# 04 — Functions: Parameters, Scope, and First-Class Behavior

Goal: master Python function mechanics: calling conventions, scope rules, closures, and functional patterns.

_Generated: 2026-02-19_

## Setup

This course targets **Python 3.11+** (works on 3.10+, with a few feature differences).

Recommended tooling:

```bash
# create + activate a virtual environment
python -m venv .venv
# mac/linux:
source .venv/bin/activate
# windows (PowerShell):
# .venv\Scripts\Activate.ps1

python -m pip install -U pip

# quality-of-life (optional but recommended)
python -m pip install -U ipykernel ruff black pytest mypy
```

If you're using Jupyter:
```bash
python -m ipykernel install --user --name python-course --display-name "Python Course (.venv)"
```

In [ ]:

import sys, platform, os
print("python:", sys.version.split()[0])
print("implementation:", platform.python_implementation())
print("platform:", platform.platform())
print("cwd:", os.getcwd())


## 1.
L1: Defining and calling functions

Key properties:
- Functions are objects.
- They can be assigned to variables, stored in containers, passed as arguments, returned.

In [ ]:

def add(a: int, b: int) -> int:
    """Return the sum of a and b."""
    return a + b

print(add(2, 3))
print(add(a=2, b=3))  # keyword arguments


## 2.
L2: Parameters: defaults, `*args`, `**kwargs`

- Defaults are evaluated **once**, at function definition time (important!).
- `*args` collects extra positional arguments into a tuple.
- `**kwargs` collects extra keyword arguments into a dict.

In [ ]:

def join(*parts: str, sep: str = " ") -> str:
    return sep.join(parts)

print(join("a", "b", "c"))
print(join("a", "b", "c", sep="-"))

def show(**kwargs):
    for k, v in kwargs.items():
        print(k, "=", v)

show(name="Ada", lang="Python")


## 3.
L3: The “mutable default” pitfall (and the fix)

Never do `def f(x=[]): ...` unless you *really* want shared state.
Use `None` and create a new list/dict inside.

In [ ]:

def append_bad(x, bucket=[]):  # DON'T (shared across calls)
    bucket.append(x)
    return bucket

def append_good(x, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(x)
    return bucket

print(append_bad(1))
print(append_bad(2))   # surprise: [1, 2]

print(append_good(1))
print(append_good(2))  # ok: [2]


## 4.
L4: Positional-only and keyword-only parameters

Python lets you control calling style:

- `def f(a, /, b)` → `a` is positional-only
- `def f(*, a)` → `a` is keyword-only

These are crucial for API design.

In [ ]:

def api(a, /, b, *, c):
    return a, b, c

print(api(1, 2, c=3))
# api(a=1, b=2, c=3)  # would fail: 'a' is positional-only


## 5.
L5: Scope: LEGB rule

Name lookup order:
- **L**ocal
- **E**nclosing (closures)
- **G**lobal (module)
- **B**uiltins

Use `nonlocal` and `global` sparingly.

In [ ]:

def make_counter():
    count = 0
    def inc():
        nonlocal count
        count += 1
        return count
    return inc

c = make_counter()
print(c(), c(), c())


## 6.
L6: Higher-order functions and callables

Common stdlib patterns:
- `sorted(iterable, key=...)`
- `map`, `filter` (often replaced by comprehensions)
- `functools.partial`

In [ ]:

from functools import partial

def power(base: int, exp: int) -> int:
    return base ** exp

square = partial(power, exp=2)
print(square(5))

words = ["banana", "fig", "apple", "kiwi"]
print(sorted(words, key=len))


## 7.
L7: Exercises

1. Write `clamp(x, lo, hi)` that bounds a number.
2. Write `compose(f, g)` returning a function that computes `f(g(x))`.
3. Implement `memoize` for a pure function (hint: dict cache).

In [ ]:

from collections.abc import Callable

def clamp(x: float, lo: float, hi: float) -> float:
    return max(lo, min(hi, x))

assert clamp(5, 0, 10) == 5
assert clamp(-1, 0, 10) == 0
assert clamp(11, 0, 10) == 10

def compose(f: Callable, g: Callable) -> Callable:
    def h(x):
        return f(g(x))
    return h

double = lambda x: 2*x
inc = lambda x: x+1
h = compose(double, inc)  # double(inc(x))
assert h(3) == 8
print("ok")


## 8.
L8: Recursion (use carefully)

Python recursion is limited (default recursion limit ~1000).
Use recursion when it makes logic simpler, otherwise use loops.

In [ ]:

def factorial(n: int) -> int:
    if n < 0:
        raise ValueError("n must be >= 0")
    if n in (0, 1):
        return 1
    return n * factorial(n - 1)

print(factorial(6))


## 9.
L9: Docstrings and introspection

Docstrings are available at runtime as `__doc__`.
Annotations are stored in `__annotations__`.

In [ ]:

import inspect

print(add.__doc__)
print(add.__annotations__)
print("signature:", inspect.signature(add))


## 10.
L10: Generators as functions (preview)

A function containing `yield` returns a generator (an iterator).

In [ ]:

def countdown(n: int):
    while n > 0:
        yield n
        n -= 1

g = countdown(3)
print(next(g), next(g), next(g))
